In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
import os
import matplotlib.colors as mcolors

In [3]:
folder_path='/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/test_death_prob/networks/delay_constant_alive_v2'

os.chdir(folder_path)

In [9]:
def create_combined_network_visualization(graph, filename, node_size=38, edge_width=14, min_filament_length=3):
    """
    Create a network visualization showing both motifs and filamentous branches.
    
    Args:
        graph: NetworkX graph object
        filename: Output filename for the visualization
        node_size: Size of nodes in the visualization
        edge_width: Width of edges in the visualization
        min_filament_length: Minimum length for filament detection
    """
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    # Identify motif nodes (nodes with 2 or more degree-1 neighbors)
    motif_nodes = [node for node in graph.nodes() 
                   if sum(1 for neighbor in graph.neighbors(node) 
                         if graph.degree(neighbor) == 1) >= 2]
    
    # Identify degree-1 nodes connected to motif centers
    motif_degree1_nodes = set()
    for motif_node in motif_nodes:
        for neighbor in graph.neighbors(motif_node):
            if graph.degree(neighbor) == 1:
                motif_degree1_nodes.add(neighbor)
    
    # Function to identify filament nodes
    def identify_filament_nodes(graph, min_length=3):
        filament_nodes = set()
        visited = set()
        
        def traverse_filament(node):
            path = [node]
            current = node
            visited.add(current)
            
            while True:
                neighbors = list(graph.neighbors(current))
                unvisited_neighbors = [n for n in neighbors if n not in visited]
                
                if len(unvisited_neighbors) == 1 and graph.degree(unvisited_neighbors[0]) <= 2:
                    next_node = unvisited_neighbors[0]
                    path.append(next_node)
                    visited.add(next_node)
                    current = next_node
                else:
                    break
            
            return path if len(path) >= min_length else []
        
        for node, degree in dict(graph.degree()).items():
            if degree == 1 and node not in visited:
                filament = traverse_filament(node)
                filament_nodes.update(filament)
        
        return filament_nodes
    
    # Identify filament nodes
    filament_nodes = identify_filament_nodes(graph, min_filament_length)
    
    # Remove motif-related nodes from filament nodes to avoid overlap
    filament_nodes = filament_nodes - set(motif_nodes) - motif_degree1_nodes
    
    # Color nodes based on their classification
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = node_size
        node['label'] = ''
        
        # Check if the node is "alive"
        node_alive = graph.nodes[node_id].get('alive', True)  # Default to True if property doesn't exist
        
        # Determine the base color based on classification
        if node_id in motif_nodes:
            # Motif center nodes - red
            base_color = 'red'
        elif node_id in motif_degree1_nodes:
            # Degree-1 nodes connected to motif centers - orange
            base_color = 'orange'
        elif node_id in filament_nodes:
            # Filament nodes - blue
            base_color = 'blue'
        else:
            # Other nodes - dark gray
            base_color = 'darkgray'
        
        # Set node color and border
        node['color'] = base_color
        
        if not node_alive:
            # Node is not alive - add black border
            node['borderWidth'] = 5
            node['borderWidthSelected'] = 5
            node['color'] = {
                'background': base_color,
                'border': 'black'
            }
    
    # Color edges based on their classification
    for edge in nt.edges:
        from_node = edge['from']
        to_node = edge['to']
        
        # Check if edge is part of a motif (connects motif center to degree-1 node)
        is_motif_edge = (from_node in motif_nodes and to_node in motif_degree1_nodes) or \
                       (to_node in motif_nodes and from_node in motif_degree1_nodes)
        
        # Check if edge is part of a filament (connects two filament nodes)
        is_filament_edge = from_node in filament_nodes and to_node in filament_nodes
        
        if is_motif_edge:
            edge['color'] = 'red'
            edge['width'] = edge_width + 4  # Slightly thicker for motif edges
        elif is_filament_edge:
            edge['color'] = 'blue'
            edge['width'] = edge_width + 4  # Slightly thicker for filament edges
        else:
            edge['color'] = 'darkgray'
            edge['width'] = edge_width
    
    # Configure visualization settings
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

In [10]:
for temp_delay in ['-30', '0', '+30']:
    for temp_prob in ['0', '15', '30']:
        network_file='../delay_constant_alive/network_n200_delay_'+temp_delay+'_prob_'+temp_prob+'.graphml'
        html_file='network_n200_delay_'+temp_delay+'_prob_'+temp_prob+'_combined_viz.html'
        create_combined_network_visualization(nx.read_graphml(network_file), html_file, node_size=36, edge_width=18, min_filament_length=3)